# Label-SHAP: gold AGN vs. non-AGN, both feature spaces

Additive to `src/interpretation.py`'s cluster-explainer SHAP (which explains HDBSCAN cluster membership -- partly circular, since clusters are a function of the same features). This notebook trains directly on the gold label (`agn_positive` vs `agn_negative`), excluding silver/unlabeled rows entirely from training. See `results/LABEL_SHAP.md` for the full writeup; this notebook is the reproducible driver behind it. Does not modify `interpretation.py` or `analysis.ipynb`.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from src import preprocessing, label_shap
from src.conversions import (
    ZERO_POINT_JY,
    build_feature_matrix,
    build_feature_matrix_with_fluxes_and_variability,
    combine_first_column,
)

DATA_DIR = Path("../data/raw")
RESULTS_DIR = Path("../results")

In [2]:
df = preprocessing.load_catalog(DATA_DIR / "Multiwave_UMAP_6_26_26.csv")
df = preprocessing.merge_csc_spectral_fits(df, DATA_DIR / "csc_spec_fits.csv")
df = preprocessing.merge_desi(df, DATA_DIR / "Multiwave_7_8_26.csv")
df = preprocessing.clean_types(df)
df = preprocessing.mask_magnitude_sentinels(df, ZERO_POINT_JY)
df = combine_first_column(df, "PS21P_g", "LEG21P_G", "combo_g")
df = combine_first_column(df, "PS21P_r", "LEG21P_R", "combo_r")
df = combine_first_column(df, "PS21P_z", "LEG21P_Z", "combo_z_mag")
df = preprocessing.build_labels(df)

print(f"Loaded and labeled: {df.shape}")

C:\dev\ml-directory\projects\astro\src\preprocessing.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return df.merge(spec, on="name", how="left", validate="many_to_one")


Loaded and labeled: (170196, 208)


## Variant A -- v_final (22-feat, `build_feature_matrix`)

In [3]:
df_sample_A, X_A = build_feature_matrix(df)
print(f"Variant A df_sample/X shape: {df_sample_A.shape}, {X_A.shape}")

has_direct_silver_input_A = any("csc" in c.lower() or "powlaw" in c.lower() for c in X_A.columns)
print("has_csc_spectral_fit-like column present in X_A:", has_direct_silver_input_A)

X_gold_A, y_A = label_shap.filter_gold(df_sample_A, X_A)
stats_A = label_shap.imbalance_stats(y_A)
print(f"Gold subset: {X_gold_A.shape}, stats: {stats_A}")

Variant A df_sample/X shape: (25801, 211), (25801, 22)
has_csc_spectral_fit-like column present in X_A: False
Gold subset: (13115, 22), stats: {'n_pos': 12272, 'n_neg': 843, 'ratio': 14.55753262158956, 'scale_pos_weight': 0.06869295958279009}


In [4]:
oof_pred_A, best_iterations_A = label_shap.run_cv(
    X_gold_A, y_A, stats_A["scale_pos_weight"], n_splits=5, random_state=42,
)
print("Per-fold best_iteration:", best_iterations_A)

metrics_A = label_shap.evaluate_oof(y_A, oof_pred_A)
print({k: v for k, v in metrics_A.items() if k != "confusion_matrix"})
print("Confusion matrix (rows=true [0,1], cols=pred [0,1]):")
print(metrics_A["confusion_matrix"])

Per-fold best_iteration: [14, 15, 14, 12, 15]
{'average_precision': 0.9940630801455511, 'roc_auc': 0.9277640775092371, 'brier': 0.047424247546730705}
Confusion matrix (rows=true [0,1], cols=pred [0,1]):
[[  287   556]
 [  322 11950]]


In [5]:
n_estimators_A = label_shap.choose_final_n_estimators(best_iterations_A)
print("Final model n_estimators (Variant A):", n_estimators_A)

model_A = label_shap.fit_final_model(X_gold_A, y_A, stats_A["scale_pos_weight"], n_estimators_A, random_state=42)
shap_values_A = label_shap.compute_shap(model_A, X_gold_A)
top20_A = label_shap.top_features(shap_values_A, X_gold_A, n=20)
print(top20_A.head(10).to_string())

label_shap.plot_shap_bar(
    top20_A, RESULTS_DIR / "figures" / "reduced_22feat" / "label_shap_bar.png",
    "Label-SHAP (gold AGN vs non-AGN) -- Variant A (22-feat)",
)
label_shap.plot_shap_beeswarm(
    shap_values_A, RESULTS_DIR / "figures" / "reduced_22feat" / "label_shap_beeswarm.png",
    "Label-SHAP (gold AGN vs non-AGN) -- Variant A (22-feat)",
)

Final model n_estimators (Variant A): 14


             feature  mean_abs_shap
0   color_w1_over_w2       0.571003
1     xray_b_over_w1       0.215165
2      xray_b_over_r       0.183094
3     color_r_over_z       0.149530
4      xray_m_over_r       0.099502
5         WISE_w2snr       0.076652
6     color_g_over_r       0.058653
7   color_w2_over_w3       0.047525
8     xray_b_over_w2       0.036159
9  optical_r_over_w1       0.022439


## Variant B -- extended (32-feat, `build_feature_matrix_with_fluxes_and_variability`)

First driver call for this function anywhere in the repo (per `STATE.md` §3 -- previously only exercised ad hoc).

In [6]:
df_sample_B, X_B = build_feature_matrix_with_fluxes_and_variability(df)
print(f"Variant B df_sample/X shape: {df_sample_B.shape}, {X_B.shape}")

has_direct_silver_input_B = any("csc" in c.lower() or "powlaw" in c.lower() for c in X_B.columns)
print("has_csc_spectral_fit-like column present in X_B:", has_direct_silver_input_B)

X_gold_B, y_B = label_shap.filter_gold(df_sample_B, X_B)
stats_B = label_shap.imbalance_stats(y_B)
print(f"Gold subset: {X_gold_B.shape}, stats: {stats_B}")

Variant B df_sample/X shape: (22417, 231), (22417, 32)
has_csc_spectral_fit-like column present in X_B: False
Gold subset: (11621, 32), stats: {'n_pos': 11055, 'n_neg': 566, 'ratio': 19.531802120141343, 'scale_pos_weight': 0.051198552691090006}


In [7]:
oof_pred_B, best_iterations_B = label_shap.run_cv(
    X_gold_B, y_B, stats_B["scale_pos_weight"], n_splits=5, random_state=42,
)
print("Per-fold best_iteration:", best_iterations_B)

metrics_B = label_shap.evaluate_oof(y_B, oof_pred_B)
print({k: v for k, v in metrics_B.items() if k != "confusion_matrix"})
print("Confusion matrix (rows=true [0,1], cols=pred [0,1]):")
print(metrics_B["confusion_matrix"])

Per-fold best_iteration: [19, 16, 18, 20, 14]
{'average_precision': 0.9946777223792429, 'roc_auc': 0.9298716823847354, 'brier': 0.03728145488954838}
Confusion matrix (rows=true [0,1], cols=pred [0,1]):
[[  259   307]
 [  332 10723]]


In [8]:
n_estimators_B = label_shap.choose_final_n_estimators(best_iterations_B)
print("Final model n_estimators (Variant B):", n_estimators_B)

model_B = label_shap.fit_final_model(X_gold_B, y_B, stats_B["scale_pos_weight"], n_estimators_B, random_state=42)
shap_values_B = label_shap.compute_shap(model_B, X_gold_B)
top20_B = label_shap.top_features(shap_values_B, X_gold_B, n=20)
print(top20_B.head(10).to_string())

label_shap.plot_shap_bar(
    top20_B, RESULTS_DIR / "figures" / "extended_32feat" / "label_shap_bar.png",
    "Label-SHAP (gold AGN vs non-AGN) -- Variant B (32-feat)",
)
label_shap.plot_shap_beeswarm(
    shap_values_B, RESULTS_DIR / "figures" / "extended_32feat" / "label_shap_beeswarm.png",
    "Label-SHAP (gold AGN vs non-AGN) -- Variant B (32-feat)",
)

Final model n_estimators (Variant B): 17


             feature  mean_abs_shap
0   color_w1_over_w2       0.631152
1     xray_b_over_w1       0.209390
2     color_r_over_z       0.162061
3      xray_b_over_r       0.106720
4      xray_m_over_r       0.100599
5    log_flux_aper_m       0.061496
6     color_g_over_r       0.056359
7   color_w2_over_w3       0.042823
8   log_flux_combo_r       0.042619
9  optical_r_over_w1       0.024280


## Side analysis: Variant A restricted to Variant B's row index

Time-boxed (≤15 min) per task scope -- isolates feature-space effect from sample-size/composition effect, since Variant A (25,801 rows) and Variant B (22,417 rows) are not the same subset. Skipped if this doesn't fall out cleanly from a plain index intersection.

In [9]:
common_idx = X_A.index.intersection(X_B.index)
print(f"Variant A rows: {len(X_A)} | Variant B rows: {len(X_B)} | intersection: {len(common_idx)}")

df_sample_A_restricted = df_sample_A.loc[common_idx]
X_A_restricted = X_A.loc[common_idx]

X_gold_Ar, y_Ar = label_shap.filter_gold(df_sample_A_restricted, X_A_restricted)
stats_Ar = label_shap.imbalance_stats(y_Ar)
print(f"Restricted gold subset: {X_gold_Ar.shape}, stats: {stats_Ar}")

oof_pred_Ar, best_iterations_Ar = label_shap.run_cv(
    X_gold_Ar, y_Ar, stats_Ar["scale_pos_weight"], n_splits=5, random_state=42,
)
metrics_Ar = label_shap.evaluate_oof(y_Ar, oof_pred_Ar)
print({k: v for k, v in metrics_Ar.items() if k != "confusion_matrix"})

n_estimators_Ar = label_shap.choose_final_n_estimators(best_iterations_Ar)
model_Ar = label_shap.fit_final_model(X_gold_Ar, y_Ar, stats_Ar["scale_pos_weight"], n_estimators_Ar, random_state=42)
shap_values_Ar = label_shap.compute_shap(model_Ar, X_gold_Ar)
top20_Ar = label_shap.top_features(shap_values_Ar, X_gold_Ar, n=20)
print(top20_Ar.head(10).to_string())

Variant A rows: 25801 | Variant B rows: 22417 | intersection: 22417
Restricted gold subset: (11621, 22), stats: {'n_pos': 11055, 'n_neg': 566, 'ratio': 19.531802120141343, 'scale_pos_weight': 0.051198552691090006}


{'average_precision': 0.995709911960646, 'roc_auc': 0.934296954034837, 'brier': 0.036767341374107164}


             feature  mean_abs_shap
0   color_w1_over_w2       0.625100
1     xray_b_over_w1       0.260391
2     color_r_over_z       0.164994
3      xray_m_over_r       0.115569
4      xray_b_over_r       0.072295
5     color_g_over_r       0.065879
6   color_w2_over_w3       0.059846
7         WISE_w2snr       0.054763
8  optical_r_over_w1       0.038503
9     xray_b_over_w2       0.017758


## Circularity check

Which top features are related to `build_labels`' silver-score inputs (Mateos wedge, legacy X-ray/optical ratio, CSC spectral-fit presence, `high_quality_z`)? See `results/LABEL_SHAP.md` for the full discussion -- this cell only confirms column presence/absence, not the relationship strength.

In [10]:
for name, X in [("A", X_A), ("B", X_B)]:
    cols = set(X.columns)
    print(
        name,
        "| color_w1_over_w2 (Mateos W1-W2 analog):", "color_w1_over_w2" in cols,
        "| color_w2_over_w3 (Mateos W2-W3 analog):", "color_w2_over_w3" in cols,
        "| xray_b_over_r (legacy ratio analog):", "xray_b_over_r" in cols,
        "| high_quality_z:", "high_quality_z" in cols,
        "| has_csc_spectral_fit / powlaw_gamma:", any("csc" in c.lower() or "powlaw" in c.lower() for c in cols),
    )

A | color_w1_over_w2 (Mateos W1-W2 analog): True | color_w2_over_w3 (Mateos W2-W3 analog): True | xray_b_over_r (legacy ratio analog): True | high_quality_z: False | has_csc_spectral_fit / powlaw_gamma: False
B | color_w1_over_w2 (Mateos W1-W2 analog): True | color_w2_over_w3 (Mateos W2-W3 analog): True | xray_b_over_r (legacy ratio analog): True | high_quality_z: False | has_csc_spectral_fit / powlaw_gamma: False
